# Who is hit hardest by Inflation?
**Team Name:** Inflators
- Sanjana Devakumaran *(sanjud)*
- Sanjna Rachakonda *(sanjnarachakonda)*
- Tommy Ayers *(tlayers21)*
- Johnson Hwang *(hjohnson22)*
## Project Introduction
This project will investigate which groups are hit the hardest by inflation in the United States. We will evaluate and compare inflation trends with wage changes across 5 income levels (quintiles) to understand how inflation affects different parts of the population. To this end, we plan on comparing CPI (consumer price index) and mean household income to discover if there has been a decline in purchasing power which in turn, leads to higher cost-of-living pressures. We will be analyzing datasets that contain data from 2005 to 2022.
## Research Questions
1. Does inflation affect low-income groups more severely than high-income ones?
2. Do wages keep up with inflation across different industries?
3. Are certain states hit harder by inflation than others?
## Potential Data Sources
- Atlanta FED - Wage Growth: https://www.atlantafed.org/research-and-data/data/wage-growth-tracker
- FRED - Historical Wages With and Without Inflation: https://fred.stlouisfed.org/tags/series?t=inflation;wages
- BEA – Regional Price Parities: https://www.bea.gov/data/prices-inflation/regional-price-parities-state-and-metro-area
- BLS - Real Earnings: https://www.bls.gov/news.release/realer.htm


# Data
- R-CPI-I: CPI amongst different household income quintiles (https://www.bls.gov/cpi/research-series/r-cpi-i.htm)

### Loading CPI Data (Income Quintile)

We begin by loading the R-CPI-I dataset, which measures Consumer Price Index (CPI) across different household income quintiles. CPI measures the average change over time in the prices paid by urban consumers for a market basket of consumer goods. Thus, this dataset allows us to compare how inflation changes over time for different income groups. We are intentionally skipping the first row because it contains context instead of column names. We use the first actual row as the header.

In [1]:
import pandas

r_cpi_i = pandas.read_excel("data/r-cpi-i-data.xlsx", skiprows=1, header=0)
r_cpi_i.head()

,Year,Month,R-CPI-I1,R-CPI-I2,R-CPI-I3,R-CPI-I4,R-CPI-I5
0,2005,12,100.000,100.000,100.000,100.000,100.000
1,2006,1,100.708,100.745,100.756,100.749,100.805
2,2006,2,100.822,100.908,100.933,100.965,101.123
3,2006,3,101.315,101.418,101.454,101.549,101.775
4,2006,4,102.123,102.268,102.290,102.413,102.594


### Reshaping CPI Data

The original dataset stores CPI values for each income quintile in separate columns in a wide format but we converted it to long format to make the data easier to analyze and visualize. Each row represents a single observation (Year, Month, Quintile). The CPI values are stored in one column instead of multiple columns. The Quintile column identifies the income group with Quintile 1 being the bottom 20% and Quintile 5 being the top 20%. Converting the dataset from wide to long format allows us to easily group, compare, and plot inflation trends across income groups.

In [2]:
# Convert r_cpi_i table to long format
dfs = []
for quintile in range(1, 6):
    temp = r_cpi_i[['Year', 'Month', f'R-CPI-I{quintile}']].copy()
    temp = temp.rename(columns={f'R-CPI-I{quintile}': 'CPI'})
    temp['Quintile'] = quintile
    dfs.append(temp)

r_cpi_i_long = pandas.concat(dfs, ignore_index=True)
r_cpi_i_long

,Year,Month,CPI,Quintile
0,2005,12,100.000,1
1,2006,1,100.708,1
2,2006,2,100.822,1
3,2006,3,101.315,1
4,2006,4,102.123,1
...,...,...,...,...
1140,2024,8,157.072,5
1141,2024,9,157.335,5
1142,2024,10,157.526,5
1143,2024,11,157.495,5


### Loading Mean Income Data by Quintile

To better understand how inflation affects different groups and answer our research question, we also loaded a dataset containing average income values for each income quintile. This data shows that even if inflation rates are similar across groups, lower-income households may be more affected because they have less income to account for rising costs. This dataset will also later allow us to compare inflation trends with income levels. The dataset is loaded directly from an Excel file with standard column headers.

In [3]:
income_quintiles = pandas.read_excel("data/mean_income_quintiles.xlsx", header=0)
income_quintiles

,Year,Income-1,Income-2,Income-3,Income-4,Income-5
0,2005,10660,27360,46300,72830,159580
1,2006,11350,28780,48220,76330,168170
2,2007,11550,29440,49970,79110,167970
3,2008,11660,29520,50130,79760,171060
4,2009,11550,29260,49530,78690,170840
5,2010,10990,28530,49170,78880,169390
6,2011,11240,29200,49840,80080,178020
7,2012,11490,29700,51180,82100,181910
8,2013,11590,30810,53740,86470,193350
9,2014,11680,31090,54040,87830,194050


### Joining CPI Data with Income Data

In order to better understand how inflation affects different income groups we combined the CPI dataset from earlier with the mean income dataset. We did a merge on the Year column because the CPI data is recorded monthly but the income data is yearly. In addition, matching by year allows us to align inflation values with their corresponding income levels. The join lets us compare inflation (CPI) with income levels, understand if lower income groups are hit harder from inflation, and prepare the data for combined analysis. We used a left join to preserve all CPI observations while adding income information.

In [4]:
joined = pandas.merge(r_cpi_i, income_quintiles, left_on='Year', right_on='Year')
joined

,Year,Month,R-CPI-I1,R-CPI-I2,R-CPI-I3,R-CPI-I4,R-CPI-I5,Income-1,Income-2,Income-3,Income-4,Income-5
0,2005,12,100.000,100.000,100.000,100.000,100.000,10660,27360,46300,72830,159580
1,2006,1,100.708,100.745,100.756,100.749,100.805,11350,28780,48220,76330,168170
2,2006,2,100.822,100.908,100.933,100.965,101.123,11350,28780,48220,76330,168170
3,2006,3,101.315,101.418,101.454,101.549,101.775,11350,28780,48220,76330,168170
4,2006,4,102.123,102.268,102.290,102.413,102.594,11350,28780,48220,76330,168170
...,...,...,...,...,...,...,...,...,...,...,...,...
200,2022,8,154.284,152.915,151.595,150.033,147.926,16120,43850,74730,119900,277300
201,2022,9,154.758,153.263,151.838,150.307,148.306,16120,43850,74730,119900,277300
202,2022,10,155.489,153.973,152.484,150.934,148.892,16120,43850,74730,119900,277300
203,2022,11,155.439,153.927,152.298,150.771,148.714,16120,43850,74730,119900,277300


### Reshaping Joined Data

After merging the CPI and income datasets, the data was still in a wide format and each quintile had its own set of columns. In order to make the dataset easier to analyze and visualize we converted it into a long format. Now, each row represents a single observation for a specific quintile, month, and year. Additionally, CPI and income values are stored in joint columns instead of separate columns per quintile. The new Quintile column identifies income group with 1 being the lowest and 5 being the highest. Converting to long format allowed us to more easily group and compare quintiles, create cleaner visualizations, and perform analysis across income groups. 

In [5]:
# Convert joined table to long format
dfs = []
for quintile in range(1, 6):
    # Select Year, Month, and the specific CPI/Income columns for each quintile
    cols = ['Year', 'Month', f'R-CPI-I{quintile}', f'Income-{quintile}']
    temp = joined[cols].copy()
    
    # Rename columns to generic names
    temp = temp.rename(columns={
        f'R-CPI-I{quintile}': 'CPI',
        f'Income-{quintile}': 'Income'
    })
    
    # Add the identifier column
    temp['Quintile'] = quintile
    dfs.append(temp)

# Combine all quintile dataframes into one long dataframe
joined_long = pandas.concat(dfs, ignore_index=True)
joined_long

,Year,Month,CPI,Income,Quintile
0,2005,12,100.000,10660,1
1,2006,1,100.708,11350,1
2,2006,2,100.822,11350,1
3,2006,3,101.315,11350,1
4,2006,4,102.123,11350,1
...,...,...,...,...,...
1020,2022,8,147.926,277300,5
1021,2022,9,148.306,277300,5
1022,2022,10,148.892,277300,5
1023,2022,11,148.714,277300,5


### Procedural Changes & Iteration Reflection

Since Part 1, our project evolved in a few important ways. First, we refined our focus to be more data driven and specific to measurable economic indicators. Initially, our topic broadly explored \"who is affected by inflation,\" but we narrowed this down to analyzing income quintiles using CPI data and comparing it with wage/income trends over time.

We also expanded our research questions by adding a regional perspective to better understand how inflation impacts different areas. In addition, we restructured our CPI dataset from a wide format into a long format, which made it easier to analyze and visualize trends across income groups. Overall, these changes helped us create a more focused, structured, and data driven analysis.

### Credit Listing

- Sanjana Devakumaran *(sanjud)*: Researched relevant datasets and annotated/documented the code for clarity and readability.
- Sanjna Rachakonda *(sanjnarachakonda)*: Led the analysis from Part 1 to Part 2, documented key findings, and contributed to additional data research.
- Tommy Ayers *(tlayers21)*: Identified and analyzed trends across income quintiles and helped interpret the results.
- Johnson Hwang *(hjohnson22)*: Found the CPI and Income datasets, joined the CPI and Income tables, reshaped the joined data, and revised the notebook based on the feedback we got from our part 1 submission.